# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import boto3
import s3fs
import matplotlib.pyplot as plt
import seaborn as sns
# Weather data library
from meteostat import Stations, Daily
from datetime import datetime

# Connect SageMaker to S3

In [ ]:
#Create S3 client using boto3
#This allows the notebook to communicate with S3
s3 = boto3.client('s3')

#Define your S3 bucket name where your datasets are stored
bucket = "vegetation-risk-project"

#list files inside bucket
response = s3.list_objects_v2(Bucket=bucket)

#Print the names of the files in the bucket
for obj in response['Contents']:
    print(obj['Key'])

The datasets used in this project were originally downloaded from public government data sources- the U.S. Forest Service (FIA), CAL FIRE wildfire records and NOAA weather data accessed through the Meteostat. After downloading, the files were manually uploaded to an Amazon S3 bucket. The datasets are accessed from Amazon S3 using SageMaker Studio notebooks for exploration and further processing.

# Load FIA(forest vegetation data) from S3

The FIA forest vegetation dataset consists of three tables: subplot information, tree characteristics, and biomass measurements. These files are stored in Amazon S3 and loaded into the SageMaker notebook for further analysis. 

In [ ]:
#file path for FIA forest datasets inside S3 bucket
subplot_path = "s3://vegetation-risk-ml/raw/forest/CA_SUBPLOT.csv"
tree_path = "s3://vegetation-risk-ml/raw/forest/CA_TREE.csv"
biomass_path = "s3://vegetation-risk-ml/raw/forest/CA_TREE_REGIONAL_BIOMASS.csv"

In [ ]:
# Load the CSV file from S3 into a pandas DataFrame
subplot = pd.read_csv(subplot_path, low_memory=False)
print(subplot.head(1))
trees = pd.read_csv(tree_path, low_memory=False)
print(trees.head(1))
biomass = pd.read_csv(biomass_path, low_memory=False)
print(biomass.head())

In [ ]:
#basic data structure
print(subplot.info())
print(trees.info())
print(biomass.info())

How many tree records exist?

What variables describe tree characteristics?

In [ ]:
#Merge sublotplot and trees dataframes on plot identifier
combined_forest = trees.merge(subplot,on="PLT_CN",how="left")

#Merge the resulting dataframe with biomass dataframe on tree identifier
combined_forest = combined_forest.merge(biomass,left_on="CN",right_on="TRE_CN",how="left")

#Check the resulting dataframe
print("Merged forest dataset shape:", combined_forest.shape)
combined_forest.head()

The three FIA datasets are merged to create a unified forest vegetation dataset. The merge connects tree measurements with subplot environmental characteristics and biomass estimates.

In [ ]:
#Save this merged dataframe to S3 for future use
combined_forest.to_csv("s3://vegetation-risk-ml/forest/combined_forest_data.csv", index=False)

### What is the distribution of vegetation density and biomass?

In [ ]:

combined_forest[['DIA','DRYBIO_AG']].hist(bins=30)

# Load CALFIRE dataset from S3

The CAL FIRE dataset contains wildfire perimeter records for California, originaly downloaded from public website and uploaded manully into S3 in raw/fire folder.This dataset provides information on fire location, cause, start date, and burned area.

In [ ]:
# Path to wildfire dataset stored in S3
fire_path = "s3://vegetation-risk-ml/raw/fire/california_fire.csv"

# Load wildfire dataset
fire = pd.read_csv(fire_path)

# Display dataset shape and rows
print("Fire dataset shape:", fire.shape)
fire.head(3)

In [ ]:
#Basic structure of fire dataset
print(fire.info())

In [ ]:
#missing values in fire dataset
print(fire.isnull().sum())

In [ ]:
#duplicated values in fire dataset
print(fire.duplicated().sum())

# Load Weather Data Using Meteostat

Weather stations across California are retrieved using the Meteostat API. These stations provide historical weather measurements such as temperature and precipitation.

In [ ]:
# Retrieve weather stations located in California
stations = Stations()

stations = stations.region('US', 'CA')
stations = stations.fetch(100)

print(stations.head())

# download weather data
start = datetime(2015,1,1)
end = datetime(2026,1,1)

weather_data = []

for station_id in stations.index[:50]:
    
    data = Daily(station_id, start, end)
    data = data.fetch()
    
    data['station'] = station_id
    
    weather_data.append(data)

weather = pd.concat(weather_data)

weather.reset_index(inplace=True)

weather.head()

In [ ]:
#Summary statistics of weather data
print(weather.describe())

In [ ]:
#missing values
print(weather.isnull().sum())

In [ ]:
# save the weather data to S3 for future use
weather.to_csv("s3://vegetation-risk-ml/weather/california_weather_data.csv", index=False)

# DATA EXPLORATION

### What are the distributions and relationships among key weather variables (temperature, precipitation, and wind speed) across California?
    

In [ ]:

weather_subset = weather[["tavg","prcp","wspd"]]

sns.pairplot(weather_subset)

plt.suptitle("Distribution and Relationships of Weather Variables", y=1.02)
plt.show()

The pairplot visualization shows the distributions of temperature, precipitation, and wind speed, as well as their relationships. Understanding these climate variables is important because weather conditions influence vegetation growth, dryness, and wildfire spread.

### Is wildfire size distribution skewed toward extreme events?

In [ ]:
sns.histplot(fire["GIS Calculated Acres"], bins=30)
plt.title("Distribution of Wildfire Size")

Plot shows Wildfire data is right-skewed.A skewed wildfire size distribution indicates that while most fires are small, a few extremely large fires dominate total burned area.

### How do vegetation biomass, wildfire size, and temperature interact to influence vegetation risk?

In [ ]:
plt.figure(figsize=(8,6))

scatter = plt.scatter(combined_forest["REGIONAL_DRYBIOT"],fire["GIS Calculated Acres"],
    c=weather["tavg"],cmap="viridis")

plt.xlabel("Vegetation Biomass")
plt.ylabel("Wildfire Size")
plt.title("Relationship Between Biomass, Fire Size, and Temperature")

plt.colorbar(scatter, label="Average Temperature")

This visualization explores how vegetation biomass relates to wildfire size while also considering temperature conditions. Higher temperatures combined with large vegetation biomass may indicate areas with increased wildfire risk.If biomass increases with fire size, it suggests that vegetation fuel accumulation may contribute to more severe wildfires.High biomass areas represent larger vegetation fuel loads, which may increase wildfire risk.Higher biomass means more combustible vegetation.

### Where are wildfires geographically concentrated across California?

In [ ]:
plt.scatter(fire["Longitude"], fire["Latitude"], alpha=0.5)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographic Distribution of Wildfires")

This visualization helps identify geographic regions with higher wildfire activity, which may correspond to areas with higher vegetation fuel loads.